In [1]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Planetoid(root=root, name=dataset_name)
    elif split_type == "70:15:15":
        transform = RandomNodeSplit(split="train_rest", num_val=0.15, num_test=0.15)
        dataset = Planetoid(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Cora"
SPLIT_TYPE = "70:15:15"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 11:42:23,559] A new study created in memory with name: no-name-cda76bd3-2d79-4505-ab48-bd75f26d0ba2



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 11:53:47,651] Trial 0 finished with value: 0.9006567994753519 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.9006567994753519.
[I 2026-09-22 11:54:01,250] Trial 1 finished with value: 0.894909679889679 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.9006567994753519.
[I 2026-09-22 11:54:06,428] Trial 2 finished with value: 0.9047618905703226 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.9047618905703226.
[I 2026-09-22 11:54:15,374] Trial 3 finished with value: 0.8981937368710836 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.90476

[I 2026-09-22 11:57:50,886] A new study created in memory with name: no-name-8eed2e76-86af-48d5-8858-c0d2aa62ab9a


GCNII: 0.8897 +/- 0.0100

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 11:58:00,129] Trial 0 finished with value: 0.9014778137207031 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9014778137207031.
[I 2026-09-22 11:58:08,399] Trial 1 finished with value: 0.8990147511164347 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9014778137207031.
[I 2026-09-22 11:58:16,320] Trial 2 finished with value: 0.9031198422114054 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 2 with value: 0.9031198422114054.
[I 2026-09-22 11:58:21,350] Trial 3 finished with value: 0.8924466172854105 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.903119842211

In [2]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Planetoid(root=root, name=dataset_name)
    elif split_type == "70:15:15":
        transform = RandomNodeSplit(split="train_rest", num_val=0.15, num_test=0.15)
        dataset = Planetoid(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "CiteSeer"
SPLIT_TYPE = "70:15:15"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 12:02:29,805] A new study created in memory with name: no-name-3249a569-9faa-4386-8a5c-8f038da351c9



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 12:22:38,516] Trial 0 finished with value: 0.7815631429354349 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.7815631429354349.
[I 2026-09-22 12:22:48,742] Trial 1 finished with value: 0.7808951338132223 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.7815631429354349.
[I 2026-09-22 12:22:53,378] Trial 2 finished with value: 0.7875751654307047 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.7875751654307047.
[I 2026-09-22 12:23:02,140] Trial 3 finished with value: 0.7828991611798605 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.7875

[I 2026-09-22 12:29:23,614] A new study created in memory with name: no-name-985bb143-cabf-409c-b5b7-210c8960ee7e


GCNII: 0.7800 +/- 0.0147

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 12:29:46,447] Trial 0 finished with value: 0.7835671305656433 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.7835671305656433.
[I 2026-09-22 12:30:01,122] Trial 1 finished with value: 0.7835671305656433 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.7835671305656433.
[I 2026-09-22 12:30:14,492] Trial 2 finished with value: 0.7889111638069153 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 2 with value: 0.7889111638069153.
[I 2026-09-22 12:30:25,864] Trial 3 finished with value: 0.7802271445592245 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.788911163806

In [3]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Planetoid(root=root, name=dataset_name)
    elif split_type == "70:15:15":
        transform = RandomNodeSplit(split="train_rest", num_val=0.15, num_test=0.15)
        dataset = Planetoid(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "PubMed"
SPLIT_TYPE = "70:15:15"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 12:37:08,844] A new study created in memory with name: no-name-d2b3f6c2-7acd-4b1e-b31e-b5311be02a55



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 13:37:53,380] Trial 0 finished with value: 0.8584628899892172 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.8584628899892172.
[I 2026-09-22 13:40:13,380] Trial 1 finished with value: 0.8856208920478821 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.8856208920478821.
[I 2026-09-22 13:40:37,653] Trial 2 finished with value: 0.8759296735127767 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 1 with value: 0.8856208920478821.
[I 2026-09-22 13:40:57,339] Trial 3 finished with value: 0.8610547582308451 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.8856

[I 2026-09-22 14:40:21,700] A new study created in memory with name: no-name-4ed7640f-7677-4ecb-866d-e49108585c9c


GCNII: 0.8851 +/- 0.0063

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 14:41:21,692] Trial 0 finished with value: 0.8669145703315735 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8669145703315735.
[I 2026-09-22 14:43:06,115] Trial 1 finished with value: 0.8625197013219198 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8669145703315735.
[I 2026-09-22 14:43:34,419] Trial 2 finished with value: 0.8595897952715555 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8669145703315735.
[I 2026-09-22 14:44:23,136] Trial 3 finished with value: 0.9060175617535909 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 3 with value: 0.906017561753

In [4]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Planetoid(root=root, name=dataset_name)
    elif split_type == "80:10:10":
        transform = RandomNodeSplit(split="train_rest", num_val=0.10, num_test=0.10)
        dataset = Planetoid(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Cora"
SPLIT_TYPE = "80:10:10"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 15:16:17,302] A new study created in memory with name: no-name-7a06f578-73c1-40bf-9e25-bb6f94e4697c



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 15:25:27,562] Trial 0 finished with value: 0.9028290311495463 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.9028290311495463.
[I 2026-09-22 15:25:38,297] Trial 1 finished with value: 0.9028290510177612 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9028290510177612.
[I 2026-09-22 15:25:42,194] Trial 2 finished with value: 0.9077490965525309 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.9077490965525309.
[I 2026-09-22 15:25:47,758] Trial 3 finished with value: 0.9015990296999613 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.9077

[I 2026-09-22 15:30:08,014] A new study created in memory with name: no-name-627f6f2c-d0ed-4bc0-b39e-74a612e772bc


GCNII: 0.8993 +/- 0.0138

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 15:30:13,235] Trial 0 finished with value: 0.9003690083821615 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9003690083821615.
[I 2026-09-22 15:30:21,189] Trial 1 finished with value: 0.9003690083821615 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9003690083821615.
[I 2026-09-22 15:30:25,778] Trial 2 finished with value: 0.907749076684316 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 2 with value: 0.907749076684316.
[I 2026-09-22 15:30:31,213] Trial 3 finished with value: 0.8991390069325765 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.90774907668431

In [5]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Planetoid(root=root, name=dataset_name)
    elif split_type == "80:10:10":
        transform = RandomNodeSplit(split="train_rest", num_val=0.10, num_test=0.10)
        dataset = Planetoid(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "CiteSeer"
SPLIT_TYPE = "80:10:10"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 15:33:33,654] A new study created in memory with name: no-name-90f4e4fe-08a1-4152-926f-2d3a57377584



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 15:46:57,588] Trial 0 finished with value: 0.7897897958755493 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.7897897958755493.
[I 2026-09-22 15:47:05,261] Trial 1 finished with value: 0.7897897958755493 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.7897897958755493.
[I 2026-09-22 15:47:09,269] Trial 2 finished with value: 0.7997997999191284 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.7997997999191284.
[I 2026-09-22 15:47:15,450] Trial 3 finished with value: 0.7887887954711914 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.7997

[I 2026-09-22 15:49:54,825] A new study created in memory with name: no-name-a211677f-436a-4ebf-bef6-eac674c6a688


GCNII: 0.7733 +/- 0.0273

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 15:50:09,036] Trial 0 finished with value: 0.7917917966842651 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.7917917966842651.
[I 2026-09-22 15:50:29,508] Trial 1 finished with value: 0.7917917966842651 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.7917917966842651.
[I 2026-09-22 15:50:41,204] Trial 2 finished with value: 0.7977977991104126 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 2 with value: 0.7977977991104126.
[I 2026-09-22 15:50:51,321] Trial 3 finished with value: 0.7867867946624756 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.797797799110